# Bootstrap Sampling — Guia de Mentoria para Cientista de Dados Sênior

> **Como usar este material:** leia cada seção, execute os códigos, responda os exercícios sem olhar a solução, e volte ao conceito se travar. O objetivo é replicar a pressão de uma entrevista técnica.

---

## 1. O Problema que o Bootstrap Resolve

Você tem **uma única amostra** e quer estimar a variabilidade de um estimador (média, mediana, correlação, R²…) — sem conhecer a distribuição da população e sem coletar mais dados.

**Ideia central:** se a sua amostra é uma boa representação da população, então amostrar *com reposição* da sua própria amostra aproxima o que você faria se pudesse amostrar da população real várias vezes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

die_vals = np.array([1, 2, 3, 4, 5, 6])

# Sampling WITH replacement — base do Bootstrap
sample_with = np.random.choice(die_vals, size=20)
print("With replacement:", sample_with)
# Valores podem repetir — é isso que permite o Bootstrap funcionar

# Sampling WITHOUT replacement — esgota a amostra
# np.random.choice(die_vals, replace=False, size=20)  # → ValueError! só temos 6 elementos
sample_without = np.random.choice(die_vals, replace=False, size=6)
print("Without replacement:", sample_without)
# Sem repetição: cada valor aparece exatamente uma vez


> **Por que `replace=True` é obrigatório no Bootstrap?**
> Com reposição, cada reamostra tem o mesmo tamanho n da original e simula uma nova coleta independente da população. Sem reposição, a reamostra seria sempre idêntica à amostra original — nenhuma variabilidade seria gerada.

---

## 2. O Mecanismo: Bootstrap Distribution

O algoritmo tem três passos:

1. Da amostra original de tamanho n, gere **B amostras bootstrap** (cada uma de tamanho n, com reposição)
2. Calcule o **estimador de interesse** em cada amostra bootstrap → gera B valores
3. Use essa **distribuição bootstrap** como aproximação da distribuição amostral do estimador

In [ ]:
def bootstrap_distribution(data, estimator_fn, B=1000, seed=42):
    """
    Gera a distribuição bootstrap de um estimador.

    Args:
        data: array com os dados originais
        estimator_fn: função que recebe um array e retorna um escalar
        B: número de amostras bootstrap
        seed: para reprodutibilidade

    Returns:
        array com B valores do estimador
    """
    rng = np.random.default_rng(seed)
    n = len(data)
    boot_estimates = np.array([
        estimator_fn(rng.choice(data, size=n, replace=True))
        for _ in range(B)
    ])
    return boot_estimates


# Exemplo: distribuição bootstrap da média de um dado
boot_means = bootstrap_distribution(die_vals, np.mean, B=5000)

print(f"Média original:          {np.mean(die_vals):.4f}")
print(f"Média das médias boot:   {np.mean(boot_means):.4f}")
print(f"Std das médias boot:     {np.std(boot_means):.4f}")
print(f"Std teórico (σ/√n):      {np.std(die_vals)/np.sqrt(len(die_vals)):.4f}")

plt.figure(figsize=(8, 4))
plt.hist(boot_means, bins=40, edgecolor='white', color='steelblue', alpha=0.85)
plt.axvline(np.mean(die_vals), color='firebrick', linewidth=2, label='Média original (3.5)')
plt.title("Distribuição Bootstrap da Média")
plt.xlabel("Média bootstrap")
plt.ylabel("Frequência")
plt.legend()
plt.tight_layout()
plt.show()


---

## 3. Intervalos de Confiança via Bootstrap

### 3.1 Método Percentile (o mais simples)

In [ ]:
def bootstrap_ci_percentile(data, estimator_fn, B=1000, alpha=0.05, seed=42):
    """IC bootstrap pelo método dos percentis."""
    boot_dist = bootstrap_distribution(data, estimator_fn, B=B, seed=seed)
    lower = np.percentile(boot_dist, 100 * alpha / 2)
    upper = np.percentile(boot_dist, 100 * (1 - alpha / 2))
    return lower, upper


# Média — comparar com IC analítico via CLT
ci_mean = bootstrap_ci_percentile(die_vals, np.mean, B=5000)
print(f"IC 95% Bootstrap (média): [{ci_mean[0]:.3f}, {ci_mean[1]:.3f}]")

# Mediana — aqui o Bootstrap brilha: CLT não tem forma fechada para a mediana
ci_median = bootstrap_ci_percentile(die_vals, np.median, B=5000)
print(f"IC 95% Bootstrap (mediana): [{ci_median[0]:.3f}, {ci_median[1]:.3f}]")


> **Ponto de entrevista:** "Quando você usaria Bootstrap em vez do CLT para construir um IC?"
> → Quando o estimador não tem forma fechada analítica (mediana, correlação, R²) **ou** quando n é pequeno e a normalidade assintótica não se sustenta.

### 3.2 Erro Padrão Bootstrap

In [ ]:
def bootstrap_se(data, estimator_fn, B=1000, seed=42):
    """Estima o erro padrão de qualquer estimador via Bootstrap."""
    boot_dist = bootstrap_distribution(data, estimator_fn, B=B, seed=seed)
    return np.std(boot_dist, ddof=1)

se_mean   = bootstrap_se(die_vals, np.mean)
se_median = bootstrap_se(die_vals, np.median)
print(f"SE bootstrap da média:   {se_mean:.4f}")
print(f"SE bootstrap da mediana: {se_median:.4f}")


---

## 4. O Teorema dos 36.8% — Conexão com Random Forest

Em cada amostra bootstrap de tamanho n, a probabilidade de um dado específico **não** ser selecionado é:

P(não selecionado) = (1 - 1/n)^n  →  e^{-1} ≈ 0.368  quando n → ∞

Ou seja, ~36.8% dos dados originais ficam fora de cada amostra bootstrap. Esses são os dados **out-of-bag (OOB)**.

In [ ]:
# Verificação empírica
n = 1000
B = 5000
rng = np.random.default_rng(42)
data = np.arange(n)

oob_fractions = []
for _ in range(B):
    boot_sample = rng.choice(data, size=n, replace=True)
    oob = set(data) - set(boot_sample)
    oob_fractions.append(len(oob) / n)

print(f"Fração OOB empírica:  {np.mean(oob_fractions):.4f}")
print(f"Valor teórico (e⁻¹): {np.exp(-1):.4f}")


> **Por que isso importa para entrevistas de ML?**
> O Random Forest usa exatamente esses dados OOB como validation set gratuito — daí o `oob_score=True` do sklearn. Saber **de onde vem** o 36.8% distingue quem usa o algoritmo de quem entende o algoritmo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
rf.fit(X, y)
print(f"OOB Score (sem cross-val): {rf.oob_score_:.4f}")


---

## 5. Bootstrap para Hypothesis Testing

Quando as premissas dos testes paramétricos não se sustentam, você pode construir a distribuição nula empiricamente.

In [ ]:
# Exemplo: testar se duas amostras têm médias diferentes
np.random.seed(42)
grupo_a = np.random.normal(loc=5.0, scale=1.5, size=40)
grupo_b = np.random.normal(loc=5.8, scale=1.5, size=40)

obs_diff = np.mean(grupo_b) - np.mean(grupo_a)
print(f"Diferença observada: {obs_diff:.4f}")

# Bootstrap test: combinar os grupos e reamostrar sob H0
combined = np.concatenate([grupo_a, grupo_b])
B = 10000
boot_diffs = []

rng = np.random.default_rng(42)
for _ in range(B):
    boot = rng.choice(combined, size=len(combined), replace=True)
    boot_a = boot[:len(grupo_a)]
    boot_b = boot[len(grupo_a):]
    boot_diffs.append(np.mean(boot_b) - np.mean(boot_a))

p_value = np.mean(np.abs(boot_diffs) >= np.abs(obs_diff))
print(f"Bootstrap p-value: {p_value:.4f}")


---

## 6. Limites do Bootstrap — O que um Sênior Precisa Saber

| Situação | Problema | Alternativa |
|---|---|---|
| n muito pequeno (< 20) | A amostra não representa bem a população | Testes exatos, prior bayesiano |
| Dados com dependência temporal | Bootstrap clássico quebra a autocorrelação | Block Bootstrap, stationary bootstrap |
| Caudas extremamente pesadas | Estimativas instáveis | Bootstrap com mais B, ou métodos robustos |
| Estimadores não-suaves (moda, quantis extremos) | Distribuição bootstrap pode ser degenerada | Smoothed bootstrap |

---

## 7. Exercícios

### Exercício 1 — Conceitual (warm-up)
Explique com suas palavras por que `replace=True` é obrigatório no Bootstrap. O que aconteceria com a distribuição bootstrap se usássemos `replace=False`?

---

### Exercício 2 — Erro Padrão da Correlação
Dado o dataset abaixo, estime o erro padrão da correlação de Pearson entre `x` e `y` usando Bootstrap com B=2000. Construa também o IC 95%.

In [ ]:
np.random.seed(7)
x = np.random.normal(0, 1, 50)
y = 0.7 * x + np.random.normal(0, 0.5, 50)

# Dica: seu estimator_fn precisa receber um array 2D (n, 2)
# e retornar np.corrcoef(arr[:,0], arr[:,1])[0,1]

# Seu código aqui


**O que observar:** compare com o IC analítico via Fisher's z-transformation. O Bootstrap deve ser próximo.

---

### Exercício 3 — OOB Teórico vs Empírico
Verifique empiricamente a convergência de (1 - 1/n)^n para e^{-1} para n ∈ {10, 50, 100, 500, 1000, 5000}. Plote a curva.

In [ ]:
ns = [10, 50, 100, 500, 1000, 5000]
# Calcule (1 - 1/n)^n para cada n e compare com e^{-1}

# Seu código aqui


---

### Exercício 4 — Bootstrap CI para uma Estatística Complexa
Calcule o IC 95% bootstrap para o **coeficiente de variação** (CV = std/mean) de uma amostra exponencial.

In [ ]:
np.random.seed(42)
amostra = np.random.exponential(scale=2.0, size=30)

def coef_variacao(arr):
    return np.std(arr, ddof=1) / np.mean(arr)

# Use bootstrap_ci_percentile com B=5000
# O CV teórico de uma Exp(λ) é sempre 1.0 — o IC deve conter esse valor?

# Seu código aqui


---

### Exercício 5 — Bootstrap Test vs t-test (nível sênior)
Compare o resultado de um bootstrap test bilateral com o t-test de Student para os grupos abaixo. Os p-values concordam? Em que situação eles discordariam?

In [ ]:
np.random.seed(99)
# Distribuição assimétrica (log-normal)
grupo_controle = np.random.lognormal(mean=0, sigma=0.5, size=35)
grupo_tratamento = np.random.lognormal(mean=0.3, sigma=0.5, size=35)

from scipy import stats
t_stat, p_ttest = stats.ttest_ind(grupo_controle, grupo_tratamento)
print(f"t-test p-value: {p_ttest:.4f}")

# Implemente o bootstrap test aqui e compare
# Seu código aqui


**Reflexão:** o t-test assume normalidade. Para dados log-normais com n=35, o CLT já ajuda — mas e se n=10? Refaça com size=10 e compare.

---

## 8. Problema de Machine Learning para Praticar

### Projeto: Avaliação Robusta de Modelos com Bootstrap no Dataset California Housing

**Contexto do problema:** em produção, reportar apenas a acurácia pontual de um modelo é insuficiente. Um cientista de dados sênior reporta **intervalos de confiança** para as métricas, especialmente quando o dataset de teste é pequeno ou quando o modelo será usado em decisões de negócio com risco.

**Objetivo:** treinar um modelo de regressão para prever o valor mediano de imóveis na Califórnia e usar Bootstrap para construir ICs das métricas de avaliação (RMSE, MAE, R²).

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 1. Carregar dados
housing = fetch_california_housing()
X, y = housing.data, housing.target

# 2. Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Treinar modelo
model = GradientBoostingRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 4. Métricas pontuais
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f"RMSE pontual: {rmse:.4f}")
print(f"MAE pontual:  {mae:.4f}")
print(f"R² pontual:   {r2:.4f}")

# ─────────────────────────────────────────────
# TAREFA 1: Construa ICs bootstrap para RMSE, MAE e R²
# Dica: reamostre pares (y_test, y_pred) com replace=True
# ─────────────────────────────────────────────

def bootstrap_metric_ci(y_true, y_pred, metric_fn, B=2000, alpha=0.05, seed=42):
    """
    Constrói IC bootstrap para uma métrica de avaliação.
    Reamostre pares (y_true, y_pred) — mantém a correspondência.
    """
    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot_metrics = []
    for _ in range(B):
        idx = rng.choice(n, size=n, replace=True)
        boot_metrics.append(metric_fn(y_true[idx], y_pred[idx]))
    lower = np.percentile(boot_metrics, 100 * alpha / 2)
    upper = np.percentile(boot_metrics, 100 * (1 - alpha / 2))
    return lower, upper, boot_metrics


# Seu código para RMSE, MAE e R² aqui

# ─────────────────────────────────────────────
# TAREFA 2: OOB Score do Random Forest vs Cross-Validation
# Compare o OOB score com o CV score de 5-fold
# ─────────────────────────────────────────────

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

rf = RandomForestRegressor(n_estimators=200, oob_score=True, random_state=42)
rf.fit(X_train, y_train)

oob_r2 = rf.oob_score_
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='r2')

print(f"\nOOB R²:           {oob_r2:.4f}")
print(f"CV R² (5-fold):   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
# Reflexão: os dois devem ser próximos — por quê?

# ─────────────────────────────────────────────
# TAREFA 3: Importância de features com IC bootstrap
# Use Bootstrap para quantificar a incerteza nas importâncias
# ─────────────────────────────────────────────
feature_names = housing.feature_names

# Dica: treine B modelos em amostras bootstrap de X_train, y_train
# e colete feature_importances_ de cada um
# Construa o IC 95% para cada feature

# Seu código aqui


**O que você vai aprender com esse projeto:**
- Como reportar incerteza em métricas de ML (crítico em entrevistas sênior)
- A conexão concreta entre Bootstrap e Random Forest via OOB
- Como quantificar a incerteza nas importâncias de features — algo que diferencia análises amadoras de profissionais
- Por que reamostrar pares `(y_true, y_pred)` em vez de só `y_test` é essencial para manter a correspondência

---

## Resumo: O que um Candidato Sênior Precisa Dominar

| Tópico | Nível esperado |
|---|---|
| Diferença with/without replacement e por que importa | Saber explicar em 30 segundos |
| Construir IC bootstrap para qualquer estimador | Implementar do zero em live coding |
| Erro padrão da mediana via Bootstrap | Exemplo concreto na ponta da língua |
| Origem do 36.8% OOB no Random Forest | Derivação simples + verificação empírica |
| Bootstrap test vs t-test — quando usar cada um | Argumentar trade-offs |
| Limites do Bootstrap (dependência temporal, n pequeno) | Citar alternativas corretas |

---

*Material elaborado para preparação para entrevistas de Cientista de Dados Sênior. Conecta Bootstrap Sampling com inferência estatística, avaliação robusta de modelos e fundamentos de ensemble learning.*